# 01 — Synthetic Dataset Generation
## FairLend-Africa Research Project

This notebook documents the synthetic dataset generation process.
The dataset simulates alternative behavioral financial data for
credit scoring in African mobile-money contexts.

**Research question:** Can behavioral financial data predict
loan repayment with sufficient accuracy and fairness to serve
as an alternative to traditional credit scoring?

In [1]:
import sys
sys.path.append("..")  # so we can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.generate_dataset import generate_dataset

%matplotlib inline
plt.rcParams["figure.dpi"] = 130

In [2]:
df = generate_dataset(n=10_000)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (10000, 21)
Columns: ['region', 'gender', 'age', 'occupation', 'monthly_txn_count', 'avg_txn_amount_usd', 'wallet_balance_trend', 'airtime_recharge_freq', 'airtime_avg_amount_usd', 'has_savings_account', 'savings_consistency_score', 'monthly_savings_usd', 'has_prior_loan', 'prior_loan_repayment_rate', 'days_late_avg', 'network_diversity_score', 'bill_payment_regularity', 'merchant_payment_count', 'loan_amount_requested_usd', 'loan_duration_weeks', 'repaid']


In [3]:
print(f"Repayment rate: {df['repaid'].mean():.2%}")
print(f"\nClass counts:\n{df['repaid'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nWhy missing? Borrowers with no prior loan history:")
print(f"  has_prior_loan == 0: {(df['has_prior_loan'] == 0).sum():,} rows")

Repayment rate: 75.53%

Class counts:
repaid
1    7553
0    2447
Name: count, dtype: int64

Missing values:
prior_loan_repayment_rate    5503
days_late_avg                5503
dtype: int64

Why missing? Borrowers with no prior loan history:
  has_prior_loan == 0: 5,503 rows


In [4]:
FEATURE_COLS = [
    "monthly_txn_count", "avg_txn_amount_usd", "wallet_balance_trend",
    "airtime_recharge_freq", "airtime_avg_amount_usd",
    "has_savings_account", "savings_consistency_score", "monthly_savings_usd",
    "has_prior_loan", "prior_loan_repayment_rate", "days_late_avg",
    "network_diversity_score", "bill_payment_regularity",
    "merchant_payment_count", "loan_amount_requested_usd", "loan_duration_weeks",
]

df[FEATURE_COLS].describe().round(3)

,monthly_txn_count,avg_txn_amount_usd,wallet_balance_trend,airtime_recharge_freq,airtime_avg_amount_usd,has_savings_account,savings_consistency_score,monthly_savings_usd,has_prior_loan,prior_loan_repayment_rate,days_late_avg,network_diversity_score,bill_payment_regularity,merchant_payment_count,loan_amount_requested_usd,loan_duration_weeks
count,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,4497.000,4497.000,10000.000,10000.000,10000.000,10000.000,10000.000
mean,16.469,58.615,0.072,13.707,3.364,0.410,28.295,11.736,0.450,0.744,11.836,48.422,60.884,5.253,371.500,12.712
std,7.589,46.293,0.682,4.869,2.179,0.492,29.835,18.448,0.497,0.149,12.257,19.439,20.405,3.281,325.310,11.247
min,1.000,4.040,-2.584,2.000,0.380,0.000,0.000,0.000,0.000,0.141,0.000,0.000,2.000,0.000,17.930,4.000
25%,11.000,28.980,-0.388,10.000,1.930,0.000,4.000,0.000,0.000,0.654,3.200,34.000,47.000,3.000,170.025,4.000
50%,15.000,45.945,0.064,13.000,2.840,0.000,11.000,0.000,0.000,0.767,8.000,48.000,63.000,5.000,281.845,8.000
75%,21.000,73.120,0.534,17.000,4.160,1.000,56.000,20.360,1.000,0.860,16.100,63.000,77.000,7.000,465.502,12.000
max,49.000,773.220,2.470,36.000,34.540,1.000,99.000,166.520,1.000,0.998,96.700,98.000,100.000,24.000,6131.890,52.000


In [5]:
import os
os.makedirs("../data/synthetic", exist_ok=True)

df.to_csv("../data/synthetic/fairlend_dataset.csv", index=False)
print(f"✓ Dataset saved: ../data/synthetic/fairlend_dataset.csv")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  File size: {os.path.getsize('../data/synthetic/fairlend_dataset.csv') / 1024:.1f} KB")

✓ Dataset saved: ../data/synthetic/fairlend_dataset.csv
  Rows: 10,000
  Columns: 21
  File size: 964.9 KB


## Dataset summary for paper

| Property | Value |
|---|---|
| Total rows | 10,000 |
| Feature columns | 16 behavioral features |
| Protected attributes | region, gender, age, occupation |
| Target variable | `repaid` (binary) |
| Positive class rate | ~75.5% |
| Missing values | `prior_loan_repayment_rate`, `days_late_avg` (MNAR — no prior loan) |
| Generation method | Logistic data generating process with calibrated coefficients |

Missing values are **structurally meaningful** (Missing Not At Random).
They are handled via median imputation inside the sklearn Pipeline.